<a href="https://colab.research.google.com/github/Lattemelia/Portforio/blob/main/demand%20forecasting1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [118]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [120]:
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [121]:
marged_df.isnull().sum()

,0
id,0
brand,0
model,0
model_year,0
milage,0
fuel_type,8466
engine,0
transmission,0
ext_col,0
int_col,0


In [117]:
marged_df

KeyboardInterrupt: 

In [116]:
# 1. 最大表示行数を一時的に制限なしにする
pd.set_option("display.max_rows", None)

# 2. 集計結果を表示（これで127行すべてが表示されます）
marged_df["engine_type"].value_counts()

KeyError: 'engine_type'

In [ ]:
#エンジン名のみ抽出
root_regex = r"(?:^\d+\.\d+HP\s+)?([^\s]+)(?:\s+(.+))?$"

# 元の engine 列から直接、一撃で2カラムに代入
marged_df[["engine_displacement", "engine_type"]] = marged_df["engine"].str.extract(root_regex)


In [ ]:
marged_df["fuel_type"].value_counts()

In [ ]:
import numpy as np
import pandas as pd


def simplify_engine_type_ultimate(engine_str):
    if pd.isna(engine_str):
        return np.nan

    engine_str = str(engine_str).lower().strip()

    # 1. 🚨【超重要】ガソリン車・内燃機関・水素車の強力な除外キーワード
    # これらが含まれていたら、どれだけ electric や motor と書いてあっても絶対に「完全なEV」にはしない
    has_ice_term = any(
        x in engine_str
        for x in [
            "gasoline",
            "gas",
            "diesel",
            "cylinder",
            "v6",
            "v8",
            "v12",
            "v10",
            "i4",
            "i-4",
            "h4",
            "i3",
            "h6",
            "i6",
            "inline",
            "rotary",
            "fuel system",  # "Electric Fuel System" はガソリン車のインジェクションのこと
            "ulev",  # 排ガス規制の基準なのでガソリン車
            "sidi",
            "dohc",
            "sohc",
            "mpfi",
            "hydrogen",  # 水素車は別扱いにするため除外
        ]
    )

    # 2. ハイブリッド（PHEV、マイルドハイブリッド含む）の判定
    is_hybrid = any(
        x in engine_str
        for x in [
            "hybrid",
            "plug-in",
            "mild electric",
            "w/range extender",  # BMW i3などの発電用エンジン付きもハイブリッド扱いへ
        ]
    )
    suffix = " Hybrid" if is_hybrid else ""

    # 3. 完全な電気自動車（EV）の判定
    # 除外ワード（ICE）やハイブリッドの単語が「無く」、かつEV系の単語がある場合のみ Electric にする
    is_ev_term = any(
        x in engine_str
        for x in [
            "electric",
            "battery",
            "70kw",
            "160kw",
            "ev",
            "bev",
            "range battery",
        ]
    )

    if is_ev_term and not has_ice_term and not is_hybrid:
        return "Electric"

    # 4. 水素燃料電池車（FCEV）は別枠にしてすっきりさせる
    if "hydrogen" in engine_str or "mirai" in engine_str:
        return "FCEV (Hydrogen)"

    # 5. 各気筒数の判定（ハイブリッドの suffix を自動でつける）
    if "8 cylinder" in engine_str or "v8" in engine_str or "8 cly" in engine_str:
        return "V8" + suffix
    if (
        "6 cylinder" in engine_str
        or "v6" in engine_str
        or "i6" in engine_str
        or "h6" in engine_str
        or "flat 6" in engine_str
        or "straight 6" in engine_str
    ):
        return "V6" + suffix
    if (
        "4 cylinder" in engine_str
        or "i4" in engine_str
        or "i-4" in engine_str
        or "h4" in engine_str
    ):
        return "I4" + suffix
    if "3 cylinder" in engine_str or "i3" in engine_str:
        return "I3" + suffix
    if "12 cylinder" in engine_str or "v12" in engine_str or "w12" in engine_str:
        return "V12" + suffix
    if "10 cylinder" in engine_str or "v10" in engine_str:
        return "V10" + suffix

    # 6. 気筒数が不明だが、ハイブリッドである場合
    if is_hybrid:
        return "Other Hybrid"

    # 7. 純EVの敗者復活（上記すべての除外に引っかからず、battery等がある場合）
    if (
        "battery" in engine_str
        or "electric" in engine_str
        or "motor - standard" in engine_str
    ):
        return "Electric"

    return "Other"


# 💡 新しい究極ロジックで列を生成
marged_df["engine_type_clean"] = marged_df["engine_type"].apply(
    simplify_engine_type_ultimate
)

# 💡 再度、Electric（EV）だけに絞り込んで車種を確認
df7_new = marged_df[marged_df["engine_type_clean"] == "Electric"]
print("✨ 【修正後】本物の電気自動車（EV）だけの車種内訳 ✨")
with pd.option_context("display.max_rows", None):
    print(df7_new.groupby("engine_type")["model"].value_counts())

In [ ]:
df7 = marged_df[marged_df["engine_type_clean"] == "Electric"]
df7.groupby("engine_type")["model"].value_counts()

In [ ]:
#エンジンがわからないかつ明らかEVを"Electric"に変換
ev_keywords_mask = marged_df["engine"].str.contains("Electric|Battery|Dual|Standard|111.2Ah", case=False, na=False)

# 排気量が NaN、かつ EV系のキーワードが含まれる行の engine_displacement を "Electric" に書き換える
marged_df.loc[marged_df["engine_displacement"].isnull() & ev_keywords_mask, "engine_displacement"] = "Electric"


In [ ]:
marged_df = marged_df.reset_index(drop=True)

#型式の誤侵入を修正
invalid_displacement_mask = marged_df["engine_displacement"].isin(
    ["V6", "I4", "V8"]
)

#インデックスを振り直したため、エラーにならずに代入できるようになります
marged_df.loc[invalid_displacement_mask, "engine_type"] = marged_df.loc[
    invalid_displacement_mask, "engine_displacement"
]

#移動させたので、排気量側は一旦 NaN（空）にする
marged_df.loc[invalid_displacement_mask, "engine_displacement"] = np.nan


#排気量を「純粋な数値」に変換する
marged_df["engine_displacement_num"] = pd.to_numeric(
    marged_df["engine_displacement"].str.replace("L", "", regex=False),
    errors="coerce",
)

In [ ]:
#engine_displacement が "Intercooled" になっている行を特定し、ディーゼルとハイオクタイプで仕分け
intercooled_mask = marged_df["engine_displacement"] == "Intercooled"

# engine 列から「数字 . 数字 + L」を抜き出す（2.0 L も 6.7 L も自動対応）
extracted_disp = (
    marged_df.loc[intercooled_mask, "engine"]
    .str.extract(r"(\d+\.\d+)\s*L")[0]
    .astype(float)
)

# 数値カラムと文字列カラムにそれぞれ代入
marged_df.loc[intercooled_mask, "engine_displacement_num"] = extracted_disp
marged_df.loc[intercooled_mask, "engine_displacement"] = np.where(
    extracted_disp.notnull(), extracted_disp.astype(str) + "L", np.nan
)

# engine 列から「I-4」「V-8」「V6」などの塊を抜き出し、ハイフンを除去する
extracted_type = (
    marged_df.loc[intercooled_mask, "engine"]
    .str.extract(r"\b([IVX]\-?\d+)\b")[0]
    .str.replace("-", "", regex=False)
)

# 抽出した型式（I4, V8 など）を engine_type に代入
marged_df.loc[intercooled_mask, "engine_type"] = extracted_type
